In [1]:
import torch
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, AutoModelForQuestionAnswering
from tqdm import tqdm
from sentence_transformers import SentenceTransformer, util

c:\Users\kisho\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
MODEL_1_PATH = "fin-qa-dapt"            # RoBERTa (The "Parrot")
MODEL_2_PATH = "deberta_sarang_plus"    # DeBERTa (The "Logician")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"🚀 Loading Ensemble on {DEVICE}...")

🚀 Loading Ensemble on cuda...


In [3]:
# Load Model 1 (RoBERTa)
tok1 = AutoTokenizer.from_pretrained(MODEL_1_PATH)
mod1 = AutoModelForQuestionAnswering.from_pretrained(MODEL_1_PATH).to(DEVICE).eval()

# Load Model 2 (DeBERTa)
# Note: DeBERTa-v3 tokenizer is SentencePiece-based, quite different from RoBERTa
tok2 = AutoTokenizer.from_pretrained(MODEL_2_PATH)
mod2 = AutoModelForQuestionAnswering.from_pretrained(MODEL_2_PATH).to(DEVICE).eval()

In [4]:
import torch
import re
sas_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
def predict_ensemble(context, question):
    # --- 1. MODEL INFERENCE ---
    in1 = tok1(question, context, return_tensors="pt", max_length=384, truncation="only_second").to(DEVICE)
    with torch.no_grad():
        out1 = mod1(**in1)
    
    in2 = tok2(question, context, return_tensors="pt", max_length=384, truncation="only_second").to(DEVICE)
    with torch.no_grad():
        out2 = mod2(**in2)

    # --- 2. CANDIDATE GENERATION ---
    candidates = {}

    def process_logits(start_logits, end_logits, tokenizer, inputs, weight):
        # Top 15 to ensure we find the clean number
        s_indices = torch.topk(start_logits, 15).indices[0]
        e_indices = torch.topk(end_logits, 15).indices[0]
        
        for s in s_indices:
            for e in e_indices:
                if e < s: continue
                if e - s > 60: continue 
                
                span_ids = inputs.input_ids[0][s:e+1]
                text = tokenizer.decode(span_ids, skip_special_tokens=True).strip()
                score = (start_logits[0][s] + end_logits[0][e]).item()
                candidates[text] = candidates.get(text, 0) + (score * weight)

    process_logits(out1.start_logits, out1.end_logits, tok1, in1, weight=0.55)
    process_logits(out2.start_logits, out2.end_logits, tok2, in2, weight=0.45)
    
    # --- 3. NEURO-SYMBOLIC GUARDRAILS ---
    best_ans = ""
    best_score = -float('inf')
    
    # A. Detect Question Type
    def is_factoid(question):
        # Define a generic "anchor" question
        anchor = "What is the financial value or amount?"
        
        # Check similarity
        sim = util.cos_sim(sas_model.encode(question), sas_model.encode(anchor))
        
        # If similarity > 0.4, it's asking for a number
        return sim > 0.4
    
    # B. Patterns
    money_pattern = re.compile(r'([$£€¥]\s?\d+(\.\d+)?\s?(billion|million|bn|m|k)?)|(\d+(\.\d+)?%)')
    
    # C. "Bad" Starts for Factoids (Explanations or Comparisons)
    bad_starts = ("marking", "reflecting", "resulting", "representing", "due to", "driven by", "caused by", "previous", "prior", "up from", "down from", "compared")

    for ans, score in candidates.items():
        ans_lower = ans.lower()
        
        # --- LOGIC FOR FACTOIDS ("How much?") ---
        if is_factoid:
            # 1. REGEX MATCH: Boost if it looks like money/percent
            has_money = money_pattern.search(ans)
            
            # 2. COMPARATIVE PENALTY (The Fix for Tesla)
            # If it says "previous year's", "up from", "down from" -> It's the wrong number.
            if any(w in ans_lower for w in ["previous", "prior", "up from", "down from", "versus", "vs"]):
                score -= 15.0 # Nuke it
            
            # 3. LENGTH RULES
            word_count = len(ans.split())
            
            if word_count > 8: 
                score -= 10.0 # Too long to be a factoid
            
            # 4. PREFER CONCISE NUMBERS
            # "$81.5 billion" (2 words) > "revenue of $81.5 billion" (4 words)
            if has_money:
                score += 5.0
                if word_count <= 3: 
                    score += 3.0 # Extra boost for purity
            
            # 5. PENALIZE DESCRIPTIONS
            if ans_lower.startswith(bad_starts):
                score -= 10.0

        # --- LOGIC FOR CAUSAL QUESTIONS ("Why?") ---
        else: 
            # Penalize "Purpose" unless asked
            if any(m in ans_lower for m in ["in order to", "aiming to"]) and "purpose" not in question.lower():
                score -= 5.0

        # General Cleanup
        if ans_lower.startswith(("that ", "which ", "the ", "a ", "an ")):
            score -= 1.0

        if score > best_score:
            best_score = score
            best_ans = ans
            
    # --- 4. FINAL POLISH ---
    best_ans = best_ans.strip(".,;: -")
    
    return best_ans

In [5]:
# ==========================================
print("Running Ensemble Evaluation...")

# Load Data & Alignment
# We load the full set and recreate the split to ensure we test on the correct 10%
df = pd.read_csv("training_data_en.csv", sep=";", engine="python")
data = []
for _, row in df.dropna(subset=["Text", "Question"]).iterrows():
    if "Answer" in row and isinstance(row["Answer"], str):
        # Only keep rows where answer is actually in text (Sarang Logic)
        if row["Text"].find(row["Answer"]) != -1:
            data.append(row)

# Re-create the validation split
val_df = pd.DataFrame(data).sample(frac=0.1, random_state=42)

preds = []
truths = val_df["Answer"].tolist()

print(f"Evaluating on {len(val_df)} aligned samples...")
for _, row in tqdm(val_df.iterrows(), total=len(val_df)):
    preds.append(predict_ensemble(row["Text"], row["Question"]))

Running Ensemble Evaluation...
Evaluating on 199 aligned samples...


100%|██████████| 199/199 [01:07<00:00,  2.96it/s]


In [6]:
sas_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
em = sum([1 if p.strip() == t.strip() else 0 for p, t in zip(preds, truths)]) / len(truths)

with torch.no_grad():
    t_emb = sas_model.encode(truths, convert_to_tensor=True)
    p_emb = sas_model.encode(preds, convert_to_tensor=True)
    sas = torch.diag(util.cos_sim(t_emb, p_emb)).mean().item()

print("\n" + "="*50)
print(f"📊 ENSEMBLE RESULTS (RoBERTa + DeBERTa):")
print(f"✅ Exact Match (EM): {em*100:.2f}%")
print(f"🧠 Semantic Score (SAS): {sas:.4f}")
print("="*50)


📊 ENSEMBLE RESULTS (RoBERTa + DeBERTa):
✅ Exact Match (EM): 87.44%
🧠 Semantic Score (SAS): 0.9424


In [13]:
# example inference
simple_context = (
    "Apple Inc. reported a net profit of $2 billion for the year 2022. "
    "The company also announced a new product line."
)
simple_question = "When did Apple report a net profit?"
answer = predict_ensemble(simple_context, simple_question)
print("Question:", simple_question)
print("Answer:", answer)

Question: When did Apple report a net profit?
Answer: $2 billion


In [10]:
sample_row = val_df.sample(n=1).iloc[0]
sample_question = sample_row["Question"]
sample_context = sample_row["Text"]
actual_answer = sample_row["Answer"]
sample_answer = predict_ensemble(sample_context, sample_question)
print("\nSample Inference from Validation Set:")
print("Question:", sample_question)
print("Answer:", sample_answer)
print("Context:", sample_context)
print("Actual Answer:", actual_answer)


Sample Inference from Validation Set:
Question: What was the implication of the stipulations set forth above?
Answer: the Committee has recommended to the Board that KPMG's re-election should be proposed to shareholders at the March 2018 AGM
Context: In light of the stipulations set forth above, the Committee has recommended to the Board that KPMG's re-election should be proposed to shareholders at the March 2018 AGM. The Committee also recommended the Board seek authority for the Directors to fix the external auditors' remuneration, having first compared the proposed fees to the prior year's fees and also relative to other companies of similar size, sector and complexity.
Actual Answer: the Committee has recommended to the Board that KPMG's re-election should be proposed to shareholders at the March 2018 AGM
